<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_34_django_forms/note_lesson_34_forms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Урок 34 — Django: форми, CRUD і PRG

| Крок | Що робимо |
|---|---|
| 0 | запускаємо `hello_project` з CRUD у ноутбуці |
| 1 | `ModelForm`: валідація і `cleaned_data` (вправи 1–2) |
| 2 | `forms.Form` для пошуку (вправа 3) |
| 3 | створення: помилки → та сама форма, успіх → redirect (вправи 4–5) |
| 4 | редагування і видалення (вправи 6–7) |
| 5 | CSRF (вправа 8) |
| 6 | знайди помилку: без PRG (вправа 9) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія, шаблони й скріншоти — у книзі курсу: [Урок 34](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_34/); поглиблено — кроки 2 і 4 [Django-книги](https://nikoriakviktot.github.io/notes_chat_app/tutorials/).

---
## 0. Проєкт у ноутбуці

Проєкт уроку — `hello_project` з формами (`hello_app/forms.py`), CRUD-views і шаблонами на Bootstrap. У Colab клітинка нижче завантажить лише папку уроку і встановить Django та crispy-forms.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_34_django_forms"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("hello_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

packages = {"django": "Django>=5.2,<6", "crispy_forms": "django-crispy-forms", "crispy_bootstrap5": "crispy-bootstrap5"}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

os.chdir("hello_project")
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 33)
django.setup()

from django.test import Client, RequestFactory
from django.test.utils import setup_test_environment

from hello_app.forms import NoteForm, NoteSearchForm
from hello_app.models import Note

setup_test_environment()                               # хост testserver і контекст шаблонів у відповідях
Note.objects.all().delete()
Note.objects.create(title="Купити молоко", content="2 л")
Note.objects.create(title="Вивчити Django Forms", content="ModelForm, clean_title, PRG", priority=3)
Note.objects.create(title="Пароль від Wi-Fi", content="coffee2026", is_pinned=True)
client = Client()
print("нотаток:", Note.objects.count())

---
## 1. ModelForm

`NoteForm(data=...)` → `is_valid()` → `cleaned_data` або `errors`. Дані з браузера — завжди **рядки**.

### Вправа 1

Перевір форму з даними `{"title": "  Здати звіт  ", "priority": "4"}`: результат `is_valid()` — у `ok`, очищені дані — у `data`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
form = NoteForm(data={"title": "  Здати звіт  ", "priority": "4"})
ok = form.is_valid()
data = form.cleaned_data
# END SOLUTION

assert ok is True
assert data["title"] == "Здати звіт", "clean_title прибирає пробіли"
assert data["priority"] == 4 and data["is_pinned"] is False, "рядок '4' → число 4"
print(data)
print("✅ Вправа 1 пройдена")

### Вправа 2

Зібери в множину `bad_fields` назви полів з помилками для даних `{"title": "", "priority": "7"}`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
bad = NoteForm(data={"title": "", "priority": "7"})
bad.is_valid()
bad_fields = set(bad.errors)
# END SOLUTION

assert bad_fields == {"title", "priority"}
for field, errors in bad.errors.items():
    print(field, "→", errors[0])
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Для порожнього заголовка спрацює `clean_title` з повідомленням «щонайменше 3 символи»?

<details>
<summary>Відповідь</summary>

Ні. Порожнє обов'язкове поле зупиняє перевірку раніше: помилка «Це поле обов'язкове.», а `clean_title` для цього поля вже не викликається.

</details>

---
## 2. forms.Form для пошуку

`NoteSearchForm` — форма без моделі з одним полем `q` (`required=False`). Пошук — `GET`, дані — з `request.GET`.

### Вправа 3

Отримай `/notes/?q=wi-fi` тестовим клієнтом і збережи заголовки нотаток з контексту шаблону (`response.context["notes"]`) у `found`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
response = client.get("/notes/", {"q": "wi-fi"})
found = [note.title for note in response.context["notes"]]
# END SOLUTION

assert found == ["Пароль від Wi-Fi"]
assert NoteSearchForm({"q": ""}).is_valid(), "порожній пошук — теж правильний"
print("✅ Вправа 3 пройдена")

---
## 3. Створення: помилки і PRG

### Вправа 4

Надішли `POST /notes/new/` з даними `{"title": "ok", "priority": "2"}`. Збережи відповідь у `bad_response`.

In [ ]:
before = Note.objects.count()
# YOUR CODE HERE
# BEGIN SOLUTION
bad_response = client.post("/notes/new/", {"title": "ok", "priority": "2"})
# END SOLUTION

assert bad_response.status_code == 200, "неправильні дані — та сама сторінка з формою"
assert "title" in bad_response.context["form"].errors
assert Note.objects.count() == before, "нічого не збережено"
print("✅ Вправа 4 пройдена")

### Вправа 5

Тепер правильні дані: `{"title": "Купити квитки", "content": "Київ — Львів", "priority": "3"}`. Відповідь — у `created`. Потім зайди за адресою з `created["Location"]` і збережи повідомлення сторінки (рядками) у `shown`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
created = client.post("/notes/new/", {"title": "Купити квитки", "content": "Київ — Львів", "priority": "3"})
page = client.get(created["Location"])
shown = [str(message) for message in page.context["messages"]]
# END SOLUTION

note = Note.objects.get(title="Купити квитки")
assert created.status_code == 302 and created["Location"] == f"/notes/{note.pk}/"
assert shown == ["Нотатку «Купити квитки» створено."]
print("✅ Вправа 5 пройдена")

**🔮 Прогноз:** Що покаже `client.get(created['Location'])` вдруге — те саме повідомлення?

<details>
<summary>Відповідь</summary>

Ні: повідомлення «на одну сторінку». Після першого показу воно зникає, а повторний `GET` нічого не створює й нічого не повідомляє.

</details>

---
## 4. Редагування і видалення

### Вправа 6

Зміни нотатку «Купити квитки» через `POST /notes/<id>/edit/`: заголовок «Купити квитки на потяг», пріоритет `4`. Відповідь — у `edited`.

In [ ]:
note = Note.objects.get(title="Купити квитки")
# YOUR CODE HERE
# BEGIN SOLUTION
edited = client.post(f"/notes/{note.pk}/edit/", {"title": "Купити квитки на потяг", "content": note.content, "priority": "4"})
# END SOLUTION

note.refresh_from_db()
assert edited.status_code == 302
assert (note.title, note.priority) == ("Купити квитки на потяг", 4)
assert Note.objects.count() == 4, "редагування не створює нову нотатку"
print("✅ Вправа 6 пройдена")

### Вправа 7

Видали нотатку «Купити молоко»: спершу `GET` сторінки видалення (код — у `confirm_code`, перевір, що нотатка ще є), потім `POST` (відповідь — у `deleted`).

In [ ]:
milk = Note.objects.get(title="Купити молоко")
url = f"/notes/{milk.pk}/delete/"
# YOUR CODE HERE
# BEGIN SOLUTION
confirm_code = client.get(url).status_code
still_there = Note.objects.filter(pk=milk.pk).exists()
deleted = client.post(url)
# END SOLUTION

assert confirm_code == 200 and still_there, "GET лише питає"
assert deleted.status_code == 302 and deleted["Location"] == "/notes/"
assert not Note.objects.filter(pk=milk.pk).exists()
print("✅ Вправа 7 пройдена")

---
## 5. CSRF

Звичайний тестовий клієнт не перевіряє CSRF-токен. `Client(enforce_csrf_checks=True)` поводиться як справжній сервер.

### Вправа 8

Надішли `POST /notes/new/` клієнтом з перевіркою CSRF і без токена. Код відповіді — у `code`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
strict = Client(enforce_csrf_checks=True)
code = strict.post("/notes/new/", {"title": "Без токена", "priority": "2"}).status_code
# END SOLUTION

assert code == 403
assert not Note.objects.filter(title="Без токена").exists()
print("✅ Вправа 8 пройдена")

---
## 6. Знайди помилку

```python
def note_create_buggy(request):
    form = NoteForm(request.POST or None)
    if request.method == "POST" and form.is_valid():
        form.save()
    return render(request, "hello_app/note_form.html", {"form": form, "heading": "Нова нотатка"})
```

Після успішного `POST` — сторінка `200`: F5 надішле форму ще раз і створить дубль.

### Вправа 9

Напиши `note_create_fixed(request)` за шаблоном PRG: успіх → `redirect("hello_app:note_detail", pk=note.pk)`, інакше — форма з помилками.

In [ ]:
from django.shortcuts import redirect, render


def note_create_fixed(request):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    form = NoteForm(request.POST or None)
    if request.method == "POST" and form.is_valid():
        note = form.save()
        return redirect("hello_app:note_detail", pk=note.pk)
    return render(request, "hello_app/note_form.html", {"form": form, "heading": "Нова нотатка"})
    # END SOLUTION


factory = RequestFactory()
response = note_create_fixed(factory.post("/notes/new/", {"title": "Оплатити інтернет", "priority": "2"}))
note = Note.objects.get(title="Оплатити інтернет")
assert response.status_code == 302 and response["Location"] == f"/notes/{note.pk}/"
assert note_create_fixed(factory.post("/notes/new/", {"title": "x", "priority": "2"})).status_code == 200
print("✅ Вправа 9 пройдена")

---
## Самоперевірка

1. Коли форма `method="get"`, а коли `method="post"`?
2. Що в `cleaned_data["priority"]` для `"3"` з браузера?
3. Опиши PRG. Яку проблему він розв'язує?
4. Чому видалення — лише `POST`?
5. Від чого захищає `{% csrf_token %}`?

<details>
<summary>Відповіді</summary>

1. `GET` — пошук і фільтри (нічого не змінює, дані в URL); `POST` — створення, зміна, видалення.
2. Число `3`: форма перетворює рядки на типи полів.
3. Post / Redirect / Get: після успішного `POST` — `302`, браузер робить `GET`. F5 повторить лише `GET`, дублів немає.
4. `GET` не має змінювати дані: посилання відкривають роботи, попереднє завантаження, розширення.
5. Від CSRF: чужий сайт не знає секретного токена сторінки, тож його підроблений `POST` отримає `403`.

</details>

## Далі

- Книга курсу: [Урок 34](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_34/) — HTML-форми, `base.html`, Bootstrap, crispy-forms, скріншоти.
- Запусти проєкт: `python manage.py runserver` у папці `hello_project` → `http://127.0.0.1:8000/notes/`.
- **Урок 35** — REST API до нотаток (DRF).